# Phase 1: Machine-Learning analysis 

This notebook replaces the previous single 80:20 evaluation used to construct Table 1.

It evaluates:

- 4 class-distribution conditions: Original, RUS, ROS, and SMOTE
- 8 baseline classifiers
- 7 TPB predictor configurations
- the complete set of video-content predictors jointly with every TPB configuration

All model comparisons are conducted only on the training set using five-times repeated stratified five-fold cross-validation. Scaling and resampling are performed inside each cross-validation training fold to prevent information leakage.

## 1. Imports and reproducibility settings

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, make_scorer
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

MODEL_SEED = 0
CV_SEED = 0
SPLIT_SEED = 20
RESAMPLING_SEED = 20

TEST_SIZE = 0.20
N_SPLITS = 5
N_REPEATS = 5

OUTPUT_DIR = Path("table1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    f"Evaluation design: {N_REPEATS} repeats × "
    f"{N_SPLITS} folds = {N_REPEATS * N_SPLITS} validation scores "
    "per model–configuration combination."
)

Evaluation design: 5 repeats × 5 folds = 25 validation scores per model–configuration combination.


## 2. Load and prepare the data

In [2]:
# The first existing path is used automatically.
# Add another candidate path when necessary.

DATA_CANDIDATES = [
    Path("DATA.xlsx"),
    Path("D:/Project/DATA.xlsx"),
]

DATA_PATH = next(
    (path for path in DATA_CANDIDATES if path.exists()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "DATA.xlsx was not found. Place it beside this notebook or "
        "edit DATA_CANDIDATES to include the correct path."
    )

df = pd.read_excel(DATA_PATH).copy()

print(f"Loaded: {DATA_PATH}")
print(f"Original shape: {df.shape}")

Loaded: DATA.xlsx
Original shape: (408, 46)


In [3]:
# Construct the binary purchase-behaviour outcome from PB1–PB4.

PB_ITEMS = ["PB1", "PB2", "PB3", "PB4"]

missing_pb_items = [column for column in PB_ITEMS if column not in df.columns]
if missing_pb_items:
    raise KeyError(f"Missing purchase-behaviour items: {missing_pb_items}")

df["PB_sum"] = df[PB_ITEMS].sum(axis=1)

# LOW = 4–12 and HIGH = 13–20
df["PB_inf"] = np.where(df["PB_sum"] <= 12, "LOW", "HIGH")

print(df["PB_inf"].value_counts())
print(df["PB_inf"].value_counts(normalize=True).round(4))

PB_inf
HIGH    298
LOW     110
Name: count, dtype: int64
PB_inf
HIGH    0.7304
LOW     0.2696
Name: proportion, dtype: float64


## 3. Define TPB and video-content predictor configurations

In [4]:
ATTD_COLUMNS = ["ATTD1", "ATTD2", "ATTD3", "ATTD4"]
PBC_COLUMNS = ["PBC1", "PBC2", "PBC3", "PBC4"]
SN_COLUMNS = ["SN1", "SN2", "SN3", "SN4"]

VIDEO_COLUMNS = [
    "V1", "V2", "V3",
    "E1", "E2", "E3",
    "H1", "H2",
    "A1", "A2", "A3",
    "T1", "T2", "T3",
    "P1", "P2", "P3",
    "EM1", "EM2", "EM3",
]

TPB_CONFIGURATIONS = {
    "ATTD & PBC & SN": ATTD_COLUMNS + PBC_COLUMNS + SN_COLUMNS,
    "ATTD & PBC": ATTD_COLUMNS + PBC_COLUMNS,
    "ATTD & SN": ATTD_COLUMNS + SN_COLUMNS,
    "PBC & SN": PBC_COLUMNS + SN_COLUMNS,
    "ATTD": ATTD_COLUMNS,
    "PBC": PBC_COLUMNS,
    "SN": SN_COLUMNS,
}

CONFIGURATION_ORDER = list(TPB_CONFIGURATIONS.keys())

ALL_REQUIRED_PREDICTORS = sorted(
    set(
        ATTD_COLUMNS
        + PBC_COLUMNS
        + SN_COLUMNS
        + VIDEO_COLUMNS
    )
)

missing_predictors = [
    column for column in ALL_REQUIRED_PREDICTORS
    if column not in df.columns
]

if missing_predictors:
    raise KeyError(
        "The following required predictors are missing: "
        f"{missing_predictors}"
    )

analysis_columns = ALL_REQUIRED_PREDICTORS + ["PB_inf"]
analysis_df = df[analysis_columns].copy()

rows_before = len(analysis_df)
analysis_df = analysis_df.dropna().reset_index(drop=True)
rows_removed = rows_before - len(analysis_df)

print(f"Analysis sample size: {len(analysis_df)}")
print(f"Rows removed because of missing values: {rows_removed}")
print(f"Number of video-content predictors: {len(VIDEO_COLUMNS)}")

Analysis sample size: 408
Rows removed because of missing values: 0
Number of video-content predictors: 20


## 4. Reserve the held-out test set

In [5]:
# Table 1 is constructed exclusively from X_train and y_train.
# X_test and y_test remain untouched for the final model evaluation.

X_all = analysis_df[ALL_REQUIRED_PREDICTORS].copy()
y_all = analysis_df["PB_inf"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=y_all,
)

class_distribution = pd.DataFrame({
    "Subset": ["Full sample", "Training set", "Held-out test set"],
    "N": [len(y_all), len(y_train), len(y_test)],
    "LOW": [
        (y_all == "LOW").sum(),
        (y_train == "LOW").sum(),
        (y_test == "LOW").sum(),
    ],
    "HIGH": [
        (y_all == "HIGH").sum(),
        (y_train == "HIGH").sum(),
        (y_test == "HIGH").sum(),
    ],
})

display(class_distribution)

assert set(X_train.index).isdisjoint(set(X_test.index))

,Subset,N,LOW,HIGH
0,Full sample,408,110,298
1,Training set,326,88,238
2,Held-out test set,82,22,60


## 5. Define the baseline classifiers and class-distribution conditions

In [6]:
# These settings reproduce the baseline algorithms used for the
# initial model comparison. Hyperparameter optimisation is performed
# later, after the best condition/model/configuration is identified.

CLASSIFIERS = {
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        random_state=MODEL_SEED,
    ),
    "DT": DecisionTreeClassifier(
        max_depth=2,
        random_state=20,
    ),
    "GB": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=1.0,
        max_depth=1,
        random_state=MODEL_SEED,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        metric="minkowski",
        p=2,
    ),
    "LR": LogisticRegression(
        max_iter=5000,
        random_state=MODEL_SEED,
    ),
    "NB": GaussianNB(),
    "RF": RandomForestClassifier(
        random_state=MODEL_SEED,
        n_jobs=1,
    ),
    "SVM": SVC(),
}

MODEL_ORDER = list(CLASSIFIERS.keys())

# None represents the original imbalanced class distribution.
CONDITIONS = {
    "Original imbalanced class distribution": {
        "prefix": "",
        "sampler": None,
    },
    "Random undersampling (RUS)": {
        "prefix": "U-",
        "sampler": RandomUnderSampler(
            random_state=RESAMPLING_SEED
        ),
    },
    "Random oversampling (ROS)": {
        "prefix": "O-",
        "sampler": RandomOverSampler(
            random_state=RESAMPLING_SEED
        ),
    },
    "Synthetic Minority Over-sampling Technique (SMOTE)": {
        "prefix": "S-",
        "sampler": SMOTE(
            random_state=RESAMPLING_SEED
        ),
    },
}

CONDITION_ORDER = list(CONDITIONS.keys())

# Scaling is fitted only on the training portion of each CV fold.
SCALED_MODELS = {"KNN", "LR", "SVM"}

def make_leakage_safe_pipeline(model_name, estimator, sampler):
    steps = [
        (
            "scaler",
            StandardScaler()
            if model_name in SCALED_MODELS
            else "passthrough",
        )
    ]

    if sampler is not None:
        steps.append(("sampler", clone(sampler)))

    steps.append(("model", clone(estimator)))

    return ImbPipeline(steps=steps)

## 6. Repeated stratified cross-validation

In [7]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=CV_SEED,
)

# Macro-averaged metrics give equal weight to the LOW and HIGH classes.
# zero_division=0 prevents undefined-metric warnings when a model does
# not predict one of the classes in a validation fold.
SCORING = {
    "accuracy": "accuracy",
    "macro_precision": make_scorer(
        precision_score,
        average="macro",
        zero_division=0,
    ),
    "macro_recall": make_scorer(
        recall_score,
        average="macro",
        zero_division=0,
    ),
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0,
    ),
}

expected_combinations = (
    len(CONDITIONS)
    * len(CLASSIFIERS)
    * len(TPB_CONFIGURATIONS)
)

print(f"Expected combinations: {expected_combinations}")
assert expected_combinations == 224


Expected combinations: 224


In [8]:
results = []

for condition_index, (condition_name, condition_spec) in enumerate(
    CONDITIONS.items()
):
    sampler = condition_spec["sampler"]
    prefix = condition_spec["prefix"]

    print(f"\nRunning condition: {condition_name}")

    for model_index, (model_name, estimator) in enumerate(
        CLASSIFIERS.items()
    ):
        pipeline = make_leakage_safe_pipeline(
            model_name=model_name,
            estimator=estimator,
            sampler=sampler,
        )

        for configuration_index, (
            configuration_name,
            tpb_columns,
        ) in enumerate(TPB_CONFIGURATIONS.items()):

            # Every TPB configuration is modelled jointly with
            # the complete video-content predictor set.
            predictor_columns = tpb_columns + VIDEO_COLUMNS

            scores = cross_validate(
                estimator=pipeline,
                X=X_train[predictor_columns],
                y=y_train,
                scoring=SCORING,
                cv=cv,
                n_jobs=-1,
                return_train_score=True,
                error_score="raise",
            )

            results.append({
                "Condition_Order": condition_index,
                "Condition": condition_name,
                "Model_Order": model_index,
                "Model": model_name,
                "Model_Label": f"{prefix}{model_name}",
                "Configuration_Order": configuration_index,
                "Configuration": configuration_name,
                "Number_of_Predictors": len(predictor_columns),

                "CV_Accuracy_Mean":
                    scores["test_accuracy"].mean(),
                "CV_Accuracy_SD":
                    scores["test_accuracy"].std(),

                "CV_Macro_Precision_Mean":
                    scores["test_macro_precision"].mean(),
                "CV_Macro_Precision_SD":
                    scores["test_macro_precision"].std(),

                "CV_Macro_Recall_Mean":
                    scores["test_macro_recall"].mean(),
                "CV_Macro_Recall_SD":
                    scores["test_macro_recall"].std(),

                "CV_Macro_F1_Mean":
                    scores["test_macro_f1"].mean(),
                "CV_Macro_F1_SD":
                    scores["test_macro_f1"].std(),

                "CV_Train_Macro_F1_Mean":
                    scores["train_macro_f1"].mean(),

                "Number_of_Validation_Scores":
                    len(scores["test_macro_f1"]),
            })

results_long = pd.DataFrame(results)

assert len(results_long) == expected_combinations
assert (
    results_long["Number_of_Validation_Scores"]
    == N_SPLITS * N_REPEATS
).all()

print(f"\nCompleted combinations: {len(results_long)}")
display(results_long.head())



Running condition: Original imbalanced class distribution

Running condition: Random undersampling (RUS)

Running condition: Random oversampling (ROS)

Running condition: Synthetic Minority Over-sampling Technique (SMOTE)

Completed combinations: 224


,Condition_Order,Condition,Model_Order,Model,Model_Label,Configuration_Order,Configuration,Number_of_Predictors,CV_Accuracy_Mean,CV_Accuracy_SD,CV_Macro_Precision_Mean,CV_Macro_Precision_SD,CV_Macro_Recall_Mean,CV_Macro_Recall_SD,CV_Macro_F1_Mean,CV_Macro_F1_SD,CV_Train_Macro_F1_Mean,Number_of_Validation_Scores
0,0,Original imbalanced class distribution,0,AdaBoost,AdaBoost,0,ATTD & PBC & SN,32,0.855879,0.024220,0.824155,0.030592,0.806271,0.046926,0.811173,0.037304,0.925659,25
1,0,Original imbalanced class distribution,0,AdaBoost,AdaBoost,1,ATTD & PBC,28,0.848503,0.034960,0.815793,0.046244,0.795742,0.055589,0.801088,0.049042,0.900522,25
2,0,Original imbalanced class distribution,0,AdaBoost,AdaBoost,2,ATTD & SN,28,0.855860,0.027964,0.833615,0.041318,0.790725,0.043700,0.805168,0.038935,0.884125,25
3,0,Original imbalanced class distribution,0,AdaBoost,AdaBoost,3,PBC & SN,28,0.861445,0.030507,0.833610,0.043269,0.815720,0.040771,0.820593,0.037057,0.919772,25
4,0,Original imbalanced class distribution,0,AdaBoost,AdaBoost,4,ATTD,24,0.849716,0.031801,0.822933,0.050102,0.785217,0.041612,0.798710,0.042642,0.861272,25


## 7. Results 

In [9]:
# Main table: mean macro-F1, reported to four decimal places.
# SD values remain available in results_long and in the Excel output.

table1_mean = (
    results_long
    .pivot_table(
        index=[
            "Condition_Order",
            "Condition",
            "Model_Order",
            "Model_Label",
        ],
        columns="Configuration",
        values="CV_Macro_F1_Mean",
        aggfunc="first",
    )
    .reindex(columns=CONFIGURATION_ORDER)
    .reset_index()
    .sort_values(
        ["Condition_Order", "Model_Order"]
    )
    .drop(columns=["Condition_Order", "Model_Order"])
    .reset_index(drop=True)
)

display(table1_mean.round(4))

Configuration,Condition,Model_Label,ATTD & PBC & SN,ATTD & PBC,ATTD & SN,PBC & SN,ATTD,PBC,SN
0,Original imbalanced class distribution,AdaBoost,0.8112,0.8011,0.8052,0.8206,0.7987,0.8053,0.7619
1,Original imbalanced class distribution,DT,0.7727,0.7773,0.7580,0.7476,0.7166,0.7735,0.7011
2,Original imbalanced class distribution,GB,0.8027,0.8080,0.7907,0.8049,0.7697,0.7950,0.7893
3,Original imbalanced class distribution,KNN,0.8199,0.8170,0.7742,0.7977,0.7624,0.7926,0.7337
4,Original imbalanced class distribution,LR,0.8052,0.7975,0.7925,0.8087,0.7768,0.7773,0.7692
5,Original imbalanced class distribution,NB,0.8461,0.8390,0.8282,0.8383,0.8204,0.8257,0.7961
6,Original imbalanced class distribution,RF,0.8432,0.8228,0.8218,0.8234,0.7952,0.8115,0.7932
7,Original imbalanced class distribution,SVM,0.8414,0.8312,0.8313,0.8221,0.8040,0.7981,0.7868
8,Random undersampling (RUS),U-AdaBoost,0.8157,0.8121,0.7843,0.8082,0.7728,0.8036,0.7594
9,Random undersampling (RUS),U-DT,0.7591,0.7619,0.7397,0.7562,0.7401,0.7683,0.7206


In [10]:
overall_best = (
    results_long
    .sort_values(
        ["CV_Macro_F1_Mean", "CV_Macro_F1_SD"],
        ascending=[False, True]
    )
    .iloc[0]
)

print("Best overall combination")
print("Condition:", overall_best["Condition"])
print("Model:", overall_best["Model_Label"])
print("TPB configuration:", overall_best["Configuration"])
print(
    "CV macro-F1:",
    f"{overall_best['CV_Macro_F1_Mean']:.4f} "
    f"± {overall_best['CV_Macro_F1_SD']:.4f}"
)

Best overall combination
Condition: Synthetic Minority Over-sampling Technique (SMOTE)
Model: S-NB
TPB configuration: ATTD & PBC & SN
CV macro-F1: 0.8511 ± 0.0362


In [11]:
# Table containing mean ± SD for checking and supplementary reporting.

table1_mean_sd = results_long.copy()

table1_mean_sd["Macro_F1_Mean_SD"] = table1_mean_sd.apply(
    lambda row: (
        f"{row['CV_Macro_F1_Mean']:.4f} "
        f"± {row['CV_Macro_F1_SD']:.4f}"
    ),
    axis=1,
)

table1_mean_sd = (
    table1_mean_sd
    .pivot_table(
        index=[
            "Condition_Order",
            "Condition",
            "Model_Order",
            "Model_Label",
        ],
        columns="Configuration",
        values="Macro_F1_Mean_SD",
        aggfunc="first",
    )
    .reindex(columns=CONFIGURATION_ORDER)
    .reset_index()
    .sort_values(
        ["Condition_Order", "Model_Order"]
    )
    .drop(columns=["Condition_Order", "Model_Order"])
    .reset_index(drop=True)
)

display(table1_mean_sd)

Configuration,Condition,Model_Label,ATTD & PBC & SN,ATTD & PBC,ATTD & SN,PBC & SN,ATTD,PBC,SN
0,Original imbalanced class distribution,AdaBoost,0.8112 ± 0.0373,0.8011 ± 0.0490,0.8052 ± 0.0389,0.8206 ± 0.0371,0.7987 ± 0.0426,0.8053 ± 0.0461,0.7619 ± 0.0471
1,Original imbalanced class distribution,DT,0.7727 ± 0.0516,0.7773 ± 0.0525,0.7580 ± 0.0520,0.7476 ± 0.0615,0.7166 ± 0.0744,0.7735 ± 0.0541,0.7011 ± 0.0567
2,Original imbalanced class distribution,GB,0.8027 ± 0.0404,0.8080 ± 0.0395,0.7907 ± 0.0502,0.8049 ± 0.0469,0.7697 ± 0.0440,0.7950 ± 0.0412,0.7893 ± 0.0526
3,Original imbalanced class distribution,KNN,0.8199 ± 0.0452,0.8170 ± 0.0589,0.7742 ± 0.0608,0.7977 ± 0.0443,0.7624 ± 0.0652,0.7926 ± 0.0548,0.7337 ± 0.0510
4,Original imbalanced class distribution,LR,0.8052 ± 0.0393,0.7975 ± 0.0475,0.7925 ± 0.0510,0.8087 ± 0.0377,0.7768 ± 0.0462,0.7773 ± 0.0496,0.7692 ± 0.0522
5,Original imbalanced class distribution,NB,0.8461 ± 0.0377,0.8390 ± 0.0378,0.8282 ± 0.0430,0.8383 ± 0.0403,0.8204 ± 0.0480,0.8257 ± 0.0413,0.7961 ± 0.0610
6,Original imbalanced class distribution,RF,0.8432 ± 0.0390,0.8228 ± 0.0474,0.8218 ± 0.0355,0.8234 ± 0.0376,0.7952 ± 0.0538,0.8115 ± 0.0553,0.7932 ± 0.0540
7,Original imbalanced class distribution,SVM,0.8414 ± 0.0415,0.8312 ± 0.0490,0.8313 ± 0.0447,0.8221 ± 0.0439,0.8040 ± 0.0590,0.7981 ± 0.0538,0.7868 ± 0.0571
8,Random undersampling (RUS),U-AdaBoost,0.8157 ± 0.0405,0.8121 ± 0.0440,0.7843 ± 0.0445,0.8082 ± 0.0453,0.7728 ± 0.0397,0.8036 ± 0.0508,0.7594 ± 0.0512
9,Random undersampling (RUS),U-DT,0.7591 ± 0.0448,0.7619 ± 0.0478,0.7397 ± 0.0527,0.7562 ± 0.0527,0.7401 ± 0.0588,0.7683 ± 0.0570,0.7206 ± 0.0475


## Supplementary Table S1: repeated-CV performance for all 224 combinations

This table reports accuracy, macro-precision, macro-recall, and macro-F1 as mean ± SD across 25 validation scores for every class-distribution condition, classifier, and predictor configuration.


In [12]:
# Supplementary Table S1 in long format:
# one row per condition × classifier × predictor configuration.

supplementary_s1 = results_long[[
    "Condition_Order",
    "Condition",
    "Model_Order",
    "Model_Label",
    "Configuration_Order",
    "Configuration",
    "Number_of_Predictors",
    "CV_Accuracy_Mean",
    "CV_Accuracy_SD",
    "CV_Macro_Precision_Mean",
    "CV_Macro_Precision_SD",
    "CV_Macro_Recall_Mean",
    "CV_Macro_Recall_SD",
    "CV_Macro_F1_Mean",
    "CV_Macro_F1_SD",
    "Number_of_Validation_Scores",
]].copy()

supplementary_s1["Accuracy_Mean_SD"] = supplementary_s1.apply(
    lambda row: (
        f"{row['CV_Accuracy_Mean']:.4f} "
        f"± {row['CV_Accuracy_SD']:.4f}"
    ),
    axis=1,
)

supplementary_s1["Macro_Precision_Mean_SD"] = supplementary_s1.apply(
    lambda row: (
        f"{row['CV_Macro_Precision_Mean']:.4f} "
        f"± {row['CV_Macro_Precision_SD']:.4f}"
    ),
    axis=1,
)

supplementary_s1["Macro_Recall_Mean_SD"] = supplementary_s1.apply(
    lambda row: (
        f"{row['CV_Macro_Recall_Mean']:.4f} "
        f"± {row['CV_Macro_Recall_SD']:.4f}"
    ),
    axis=1,
)

supplementary_s1["Macro_F1_Mean_SD"] = supplementary_s1.apply(
    lambda row: (
        f"{row['CV_Macro_F1_Mean']:.4f} "
        f"± {row['CV_Macro_F1_SD']:.4f}"
    ),
    axis=1,
)

supplementary_s1 = (
    supplementary_s1
    .sort_values([
        "Condition_Order",
        "Model_Order",
        "Configuration_Order",
    ])
    [[
        "Condition",
        "Model_Label",
        "Configuration",
        "Number_of_Predictors",
        "Accuracy_Mean_SD",
        "Macro_Precision_Mean_SD",
        "Macro_Recall_Mean_SD",
        "Macro_F1_Mean_SD",
        "Number_of_Validation_Scores",
    ]]
    .reset_index(drop=True)
)

assert len(supplementary_s1) == expected_combinations
display(supplementary_s1.head(20))


,Condition,Model_Label,Configuration,Number_of_Predictors,Accuracy_Mean_SD,Macro_Precision_Mean_SD,Macro_Recall_Mean_SD,Macro_F1_Mean_SD,Number_of_Validation_Scores
0,Original imbalanced class distribution,AdaBoost,ATTD & PBC & SN,32,0.8559 ± 0.0242,0.8242 ± 0.0306,0.8063 ± 0.0469,0.8112 ± 0.0373,25
1,Original imbalanced class distribution,AdaBoost,ATTD & PBC,28,0.8485 ± 0.0350,0.8158 ± 0.0462,0.7957 ± 0.0556,0.8011 ± 0.0490,25
2,Original imbalanced class distribution,AdaBoost,ATTD & SN,28,0.8559 ± 0.0280,0.8336 ± 0.0413,0.7907 ± 0.0437,0.8052 ± 0.0389,25
3,Original imbalanced class distribution,AdaBoost,PBC & SN,28,0.8614 ± 0.0305,0.8336 ± 0.0433,0.8157 ± 0.0408,0.8206 ± 0.0371,25
4,Original imbalanced class distribution,AdaBoost,ATTD,24,0.8497 ± 0.0318,0.8229 ± 0.0501,0.7852 ± 0.0416,0.7987 ± 0.0426,25
5,Original imbalanced class distribution,AdaBoost,PBC,24,0.8497 ± 0.0358,0.8160 ± 0.0473,0.8010 ± 0.0477,0.8053 ± 0.0461,25
6,Original imbalanced class distribution,AdaBoost,SN,24,0.8295 ± 0.0298,0.8022 ± 0.0459,0.7448 ± 0.0480,0.7619 ± 0.0471,25
7,Original imbalanced class distribution,DT,ATTD & PBC & SN,32,0.8259 ± 0.0416,0.7858 ± 0.0542,0.7663 ± 0.0514,0.7727 ± 0.0516,25
8,Original imbalanced class distribution,DT,ATTD & PBC,28,0.8296 ± 0.0419,0.7908 ± 0.0538,0.7709 ± 0.0528,0.7773 ± 0.0525,25
9,Original imbalanced class distribution,DT,ATTD & SN,28,0.8203 ± 0.0346,0.7803 ± 0.0499,0.7486 ± 0.0558,0.7580 ± 0.0520,25


## 8. Identify the strongest configuration within each condition and model

In [13]:
best_by_condition_model = (
    results_long
    .sort_values(
        [
            "Condition_Order",
            "Model_Order",
            "CV_Macro_F1_Mean",
            "CV_Macro_F1_SD",
        ],
        ascending=[True, True, False, True],
    )
    .groupby(
        ["Condition", "Model_Label"],
        sort=False,
        as_index=False,
    )
    .first()
    [[
        "Condition",
        "Model_Label",
        "Configuration",
        "Number_of_Predictors",
        "CV_Macro_F1_Mean",
        "CV_Macro_F1_SD",
    ]]
)

display(best_by_condition_model.round(4))

,Condition,Model_Label,Configuration,Number_of_Predictors,CV_Macro_F1_Mean,CV_Macro_F1_SD
0,Original imbalanced class distribution,AdaBoost,PBC & SN,28,0.8206,0.0371
1,Original imbalanced class distribution,DT,ATTD & PBC,28,0.7773,0.0525
2,Original imbalanced class distribution,GB,ATTD & PBC,28,0.8080,0.0395
3,Original imbalanced class distribution,KNN,ATTD & PBC & SN,32,0.8199,0.0452
4,Original imbalanced class distribution,LR,PBC & SN,28,0.8087,0.0377
5,Original imbalanced class distribution,NB,ATTD & PBC & SN,32,0.8461,0.0377
6,Original imbalanced class distribution,RF,ATTD & PBC & SN,32,0.8432,0.0390
7,Original imbalanced class distribution,SVM,ATTD & PBC & SN,32,0.8414,0.0415
8,Random undersampling (RUS),U-AdaBoost,ATTD & PBC & SN,32,0.8157,0.0405
9,Random undersampling (RUS),U-DT,PBC,24,0.7683,0.0570


In [14]:
best_overall = (
    results_long
    .sort_values(
        ["CV_Macro_F1_Mean", "CV_Macro_F1_SD"],
        ascending=[False, True],
    )
    .head(10)
    [[
        "Condition",
        "Model_Label",
        "Configuration",
        "Number_of_Predictors",
        "CV_Macro_F1_Mean",
        "CV_Macro_F1_SD",
        "CV_Train_Macro_F1_Mean",
    ]]
)

print("Top 10 model–configuration combinations:")
display(best_overall.round(4))

Top 10 model–configuration combinations:


,Condition,Model_Label,Configuration,Number_of_Predictors,CV_Macro_F1_Mean,CV_Macro_F1_SD,CV_Train_Macro_F1_Mean
203,Synthetic Minority Over-sampling Technique (SM...,S-NB,ATTD & PBC & SN,32,0.8511,0.0362,0.8499
147,Random oversampling (ROS),O-NB,ATTD & PBC & SN,32,0.8506,0.0394,0.8581
161,Random oversampling (ROS),O-SVM,ATTD & PBC & SN,32,0.8487,0.0323,0.9389
210,Synthetic Minority Over-sampling Technique (SM...,S-RF,ATTD & PBC & SN,32,0.8477,0.0386,1.0000
35,Original imbalanced class distribution,NB,ATTD & PBC & SN,32,0.8461,0.0377,0.8563
154,Random oversampling (ROS),O-RF,ATTD & PBC & SN,32,0.8453,0.0387,1.0000
217,Synthetic Minority Over-sampling Technique (SM...,S-SVM,ATTD & PBC & SN,32,0.8447,0.0300,0.9390
77,Random undersampling (RUS),U-KNN,ATTD & PBC & SN,32,0.8432,0.0415,0.8620
42,Original imbalanced class distribution,RF,ATTD & PBC & SN,32,0.8432,0.0390,1.0000
206,Synthetic Minority Over-sampling Technique (SM...,S-NB,PBC & SN,28,0.8415,0.0395,0.8449


## 9. Condition-level and model-level summaries

In [15]:
condition_summary = (
    results_long
    .groupby("Condition", as_index=False)
    .agg(
        Mean_CV_Macro_F1=("CV_Macro_F1_Mean", "mean"),
        SD_Across_Combinations=("CV_Macro_F1_Mean", "std"),
        Minimum_CV_Macro_F1=("CV_Macro_F1_Mean", "min"),
        Maximum_CV_Macro_F1=("CV_Macro_F1_Mean", "max"),
    )
)

model_summary = (
    results_long
    .groupby(["Condition", "Model_Label"], as_index=False)
    .agg(
        Mean_CV_Macro_F1=("CV_Macro_F1_Mean", "mean"),
        SD_Across_Configurations=("CV_Macro_F1_Mean", "std"),
        Maximum_CV_Macro_F1=("CV_Macro_F1_Mean", "max"),
    )
)

display(condition_summary.round(4))
display(model_summary.round(4))

,Condition,Mean_CV_Macro_F1,SD_Across_Combinations,Minimum_CV_Macro_F1,Maximum_CV_Macro_F1
0,Original imbalanced class distribution,0.7974,0.0303,0.7011,0.8461
1,Random oversampling (ROS),0.7997,0.0298,0.7144,0.8506
2,Random undersampling (RUS),0.7951,0.0306,0.7206,0.8432
3,Synthetic Minority Over-sampling Technique (SM...,0.7994,0.0297,0.7176,0.8511


,Condition,Model_Label,Mean_CV_Macro_F1,SD_Across_Configurations,Maximum_CV_Macro_F1
0,Original imbalanced class distribution,AdaBoost,0.8006,0.0185,0.8206
1,Original imbalanced class distribution,DT,0.7495,0.0300,0.7773
2,Original imbalanced class distribution,GB,0.7943,0.0130,0.8080
3,Original imbalanced class distribution,KNN,0.7854,0.0309,0.8199
4,Original imbalanced class distribution,LR,0.7896,0.0153,0.8087
5,Original imbalanced class distribution,NB,0.8277,0.0165,0.8461
6,Original imbalanced class distribution,RF,0.8159,0.0176,0.8432
7,Original imbalanced class distribution,SVM,0.8164,0.0203,0.8414
8,Random oversampling (ROS),O-AdaBoost,0.8064,0.0219,0.8332
9,Random oversampling (ROS),O-DT,0.7576,0.0285,0.7915


## 10. Export all results

In [17]:
excel_path = OUTPUT_DIR / "Table1_and_Supplementary_S1_repeated_CV.xlsx"
csv_path = OUTPUT_DIR / "Table1_results_long.csv"
supp_csv_path = OUTPUT_DIR / "Supplementary_Table_S1.csv"

with pd.ExcelWriter(excel_path) as writer:
    class_distribution.to_excel(
        writer,
        sheet_name="Class_Distribution",
        index=False,
    )
    results_long.to_excel(
        writer,
        sheet_name="All_224_Combinations",
        index=False,
    )
    table1_mean.to_excel(
        writer,
        sheet_name="Table1_Mean_F1",
        index=False,
    )
    table1_mean_sd.to_excel(
        writer,
        sheet_name="Table1_Mean_SD",
        index=False,
    )
    supplementary_s1.to_excel(
        writer,
        sheet_name="Supplementary_Table_S1",
        index=False,
    )
    best_by_condition_model.to_excel(
        writer,
        sheet_name="Best_By_Model",
        index=False,
    )
    best_overall.to_excel(
        writer,
        sheet_name="Top_Combinations",
        index=False,
    )
    condition_summary.to_excel(
        writer,
        sheet_name="Condition_Summary",
        index=False,
    )
    model_summary.to_excel(
        writer,
        sheet_name="Model_Summary",
        index=False,
    )

results_long.to_csv(csv_path, index=False)
supplementary_s1.to_csv(supp_csv_path, index=False)

print(f"Excel results saved to: {excel_path}")
print(f"Long-format CSV saved to: {csv_path}")
print(f"Supplementary Table S1 CSV saved to: {supp_csv_path}")


Excel results saved to: table1_outputs\Table1_and_Supplementary_S1_repeated_CV.xlsx
Long-format CSV saved to: table1_outputs\Table1_results_long.csv
Supplementary Table S1 CSV saved to: table1_outputs\Supplementary_Table_S1.csv


## Reporting note

The revised Supplementary Table S1 reports accuracy, macro-precision,
macro-recall, and macro-F1 as mean ± standard deviation across 25
validation scores obtained from five-times repeated stratified five-fold
cross-validation on the training set. Standardisation and class-imbalance
handling are conducted only within the training portion of each fold.
The held-out test set remains untouched and should be used only after
the final model, predictor configuration, resampling condition, and
hyperparameters have been selected.
